# Haul/Dump Truck: wheel-motor bearing — sensor vs camera

| | Sensor | Camera |
|---|---|---|
| Data | UODS vibration (University of Ottawa bearing rig) | XJTU-DV-Rotor event camera (lab rotor rig) |
| Classes | healthy, inner race, outer race | same three |
| Tested on | 20 recordings from bearings never used for training | 9 recordings at an unseen speed |
| Code | `Models/haul_truck.py experiment` (both branches) | |

Ball faults are left out (the sensor data recorded them with no load). Lab rigs, not truck data; compared by fault class only.

All of this machine's experiment code is in its `Models/` file; shared notebook set-up is in `notebook_helpers.py`. Run the cells from top to bottom with the **Capstone (.venv)** kernel.

In [1]:
from notebook_helpers import *

LOCAL = setup([
    ("Sensor: UODS vibration", "Raw_Data/uods_vafdc", "*.mat", 20),
    ("Camera: XJTU-DV-Rotor zip", "Raw_Data/xjtu_dv_rotor", "*.zip", 1),
])

,environment
git commit,cf5f533
python,3.11.9
kernel,project .venv
numpy,2.4.6
scipy,1.17.1
pandas,2.3.3
scikit-learn,1.9.1
opencv-python,4.14.0.94
matplotlib,3.11.2


,location,files,MB,local
data,,,,
Sensor: UODS vibration,Raw_Data/uods_vafdc,30/20,106.9,True
Camera: XJTU-DV-Rotor zip,Raw_Data/xjtu_dv_rotor,1/1,2812.1,True


All data is local: running fully offline.


## Run both branches
**Sensor:** five rounds, each hiding two whole bearings. Inner-race files carry a constant +50 offset (a recording artefact), so each window's mean is removed. **Camera:** train on 1000 rpm, test on 2000 rpm; extra check hides each camera angle.

In [2]:
before = load("haul_truck_wheel_motor_bearing.json")
if all(LOCAL.values()):
    run_script("Models/haul_truck.py", "experiment", "--refresh", tail=1)
truck = load("haul_truck_wheel_motor_bearing.json")
check(before, truck, [("sensor", "held_out_bearings", "model"), ("visual", "train_1000_test_2000_rpm", "model")], "Haul Truck")
show({"Sensor: vibration (20 recordings)": truck["sensor"]["held_out_bearings"]["model"],
      "Camera: event images (9 recordings)": truck["visual"]["train_1000_test_2000_rpm"]["model"],
      "Camera: unseen angle (18)": truck["visual"]["held_out_camera_angle"]["model"],
      "Sensor: always guess one class": truck["sensor"]["held_out_bearings"]["most_frequent_control"],
      "Camera: always guess one class": truck["visual"]["train_1000_test_2000_rpm"]["most_frequent_control"]})

[Models/haul_truck.py experiment --refresh]
  Sensor accuracy 0.900, macro F1 0.871; camera accuracy 0.444, macro F1 0.356
Haul Truck: matches the saved result


,accuracy,macro_precision,macro_recall,macro_f1
Sensor: vibration (20 recordings),0.900,0.914,0.867,0.871
Camera: event images (9 recordings),0.444,0.333,0.444,0.356
Camera: unseen angle (18),0.444,0.300,0.444,0.357
Sensor: always guess one class,0.500,0.167,0.333,0.222
Camera: always guess one class,0.333,0.111,0.333,0.167


![Haul Truck sensor vs camera](../Figures/haul_truck/haul_truck_confusions.png)

**What this means:** the sensor works (18/20 = 0.900 on unseen bearings; guessing gets 0.500). The camera does not work well here (4/9 = 0.444, one more than guessing) and never recognises inner-race faults. Different rigs and small test sets. Full write-up: [`Results/haul_truck/haul_truck_sensor_visual_comparison.md`](../Results/haul_truck/haul_truck_sensor_visual_comparison.md).

**Data (CC BY 4.0):** UODS-VAFDC, DOI 10.17632/y2px5tg92h.1; XJTU-DV, DOI 10.5281/zenodo.17447083.

## Week 5: baseline model comparison
Four model types are trained with settings fixed in advance and tested on exactly the same held-out recordings as above: a majority-class control, logistic regression, a decision tree and a random forest. No model is picked using these test results; the main model stays the same.

In [3]:
if all(LOCAL.values()):
    run_script("Models/haul_truck.py", "baselines", tail=1)
show_baselines(load("haul_truck_baselines.json"))

[Models/haul_truck.py baselines]
  Saved Results\haul_truck\haul_truck_baselines.json and Figures\haul_truck\haul_truck_baselines.png


,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
Sensor: vibration | Majority class (control),0.500,0.333,0.167,0.333,0.222
Sensor: vibration | Logistic regression,0.850,0.800,0.820,0.800,0.806
Sensor: vibration | Decision tree,0.750,0.700,0.689,0.700,0.691
Sensor: vibration | Random forest,0.900,0.867,0.914,0.867,0.871
Camera: event camera | Majority class (control),0.333,0.333,0.111,0.333,0.167
Camera: event camera | Logistic regression,0.333,0.333,0.278,0.333,0.302
Camera: event camera | Decision tree,0.444,0.444,0.528,0.444,0.473
Camera: event camera | Random forest,0.444,0.444,0.333,0.444,0.356


![Baseline comparison](../Figures/haul_truck/haul_truck_baselines.png)

## Week 6: professor's remarks (4 classes, CSV sensor model, CNN with augmentation)
- **CSV:** the sensor features are written to `Processed_Data/haul_truck_sensor_features_4class.csv` and 4-class models are trained on it.
- **Named images:** `export-images` saves the camera images as `Images/haul_truck/<class>/haul_truck_<class>_..._tNN.png` (local only).
- **CSV model choice:** the feature set and model are chosen on validation data inside the training set only, then tested once.
- **CNN:** a pretrained ResNet18 (transfer learning) and a small CNN, both with data augmentation (flip, shift, brightness, noise), tested on the same held-out recordings. Retraining takes minutes, so it is off by default; set `TRAIN_CNN = True` to retrain.
- **Two tests:** the *main* test holds out whole recordings (or whole speeds/bearings) as before. The *time split* (secondary, easier) trains on the first 60% of every recording and tests on the last 30% (10% gap); the same recordings appear in training, so part of a high score may come from recognising the recording rather than the fault.

In [4]:
TRAIN_CNN = False
if all(LOCAL.values()):
    run_script("Models/haul_truck.py", "csv", tail=2)
    run_script("Models/haul_truck.py", "csv", "--split", "time", tail=1)
    if TRAIN_CNN:
        run_script("Models/haul_truck.py", "export-images", tail=1)
        run_script("Models/haul_truck.py", "cnn-pretrained", tail=1)
        run_script("Models/haul_truck.py", "cnn-pretrained", "--split", "time", tail=1)
        run_script("Models/haul_truck.py", "cnn", tail=1)
show_week6("haul_truck")

C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


C:\Users\laksh\Downloads\GitHub\capstone-week4\.venv\Lib\site-packages\sklearn\svm\_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


[Models/haul_truck.py csv]
  Validation-selected model: accuracy 0.800  macro F1 0.741  (choices: ['base+long envelope+microphone / Extra trees', 'base+long envelope / Gradient boosting', 'base / Random forest', 'base+long envelope / Gradient boosting', 'base+long envelope / Logistic regression'])
  Saved Processed_Data\haul_truck_sensor_features_4class.csv, Results\haul_truck\haul_truck_csv_4class.json and Figures\haul_truck\haul_truck_csv_4class.png


[Models/haul_truck.py csv --split time]
  Saved Results\haul_truck\haul_truck_csv_4class_timesplit.json and Figures\haul_truck\haul_truck_csv_4class_timesplit.png


,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1
CSV sensor | validation-selected (main),0.800,0.733,0.760,0.733,0.741
CSV sensor | Majority class (control),0.500,0.250,0.125,0.250,0.167
CSV sensor | Logistic regression,0.767,0.717,0.707,0.717,0.711
CSV sensor | Random forest,0.867,0.833,0.849,0.833,0.831
"Camera images | pretrained ResNet18 (main), per recording",0.417,0.417,0.463,0.417,0.384
"Camera images | small CNN, per recording",0.250,0.250,0.094,0.250,0.136
TIME SPLIT (secondary) | CSV sensor | Random forest,1.000,1.000,1.000,1.000,1.000
TIME SPLIT (secondary) | Camera images | pretrained ResNet18,0.913,0.900,0.929,0.900,0.899


![CSV sensor model](../Figures/haul_truck/haul_truck_csv_4class.png) ![Pretrained CNN](../Figures/haul_truck/haul_truck_cnn_pretrained_4class.png)

Time split (secondary): ![CSV time split](../Figures/haul_truck/haul_truck_csv_4class_timesplit.png) ![CNN time split](../Figures/haul_truck/haul_truck_cnn_pretrained_4class_timesplit.png)

## Try your own CSV
Put the path of any CSV in `CSV_FILE` and run this cell. It can be a **feature table** (same columns as `Processed_Data/haul_truck_sensor_features_4class.csv`, optional `class` column) or a **raw recording** (one column of sensor values). Set `LABEL` to the true class to see the accuracy. Classes: healthy, inner, outer, ball. The saved model is trained on all project data, so testing on one of the project's own recordings gives an optimistic score; to test fairly, first run `python Models/haul_truck.py train-final --exclude <recording>`.

In [5]:
CSV_FILE = None   # e.g. r"C:/Users/me/Downloads/my_recording.csv"
LABEL = None      # true class if known
if CSV_FILE:
    run_script("Models/haul_truck.py", "predict", "--csv", CSV_FILE, *((["--label", LABEL] if LABEL else [])), tail=25)
else:
    print("Set CSV_FILE to a CSV path to get predictions (and accuracy if LABEL is set).")

Set CSV_FILE to a CSV path to get predictions (and accuracy if LABEL is set).
